<a href="https://colab.research.google.com/github/Ramjeet-Dixit/IITM-AIML-Rdixit/blob/main/HuggingFace%20Demo1%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Applications and Demos

This document contains code demos for applications of Transformer models.  
Each demo is standalone and can be executed in a Python environment with Hugging Face `transformers` library.

---

## Demo 1: Machine Translation (English ↔ French)

In [1]:
pip install --upgrade transformers sentencepiece

In [2]:
import transformers
print(transformers.__version__)

5.4.0


In [3]:
# Install first if needed:
# pip install transformers sentencepiece torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------- English -> French --------
model_en_fr = "Helsinki-NLP/opus-mt-en-fr"
tokenizer_en_fr = AutoTokenizer.from_pretrained(model_en_fr)
translator_en_fr = AutoModelForSeq2SeqLM.from_pretrained(model_en_fr)

text = "Machine Learning enables computers to learn from data."

inputs_en_fr = tokenizer_en_fr(text, return_tensors="pt", truncation=True)
outputs_en_fr = translator_en_fr.generate(**inputs_en_fr, max_length=40)
translated_text = tokenizer_en_fr.decode(outputs_en_fr[0], skip_special_tokens=True)

print("Input:", text)
print("Translated:", translated_text)

# -------- French -> English --------
model_fr_en = "Helsinki-NLP/opus-mt-fr-en"
tokenizer_fr_en = AutoTokenizer.from_pretrained(model_fr_en)
translator_fr_en = AutoModelForSeq2SeqLM.from_pretrained(model_fr_en)

inputs_fr_en = tokenizer_fr_en(translated_text, return_tensors="pt", truncation=True)
outputs_fr_en = translator_fr_en.generate(**inputs_fr_en, max_length=40)
back_translated_text = tokenizer_fr_en.decode(outputs_fr_en[0], skip_special_tokens=True)

print("Back Translated:", back_translated_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Input: Machine Learning enables computers to learn from data.
Translated: Le Machine Learning permet aux ordinateurs d'apprendre à partir de données.


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Back Translated: Machine Learning allows computers to learn from data.


## Demo 2: Image Captioning

In [4]:
# Install required libraries
# pip install transformers timm sentencepiece

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests

# Load BLIP model for image captioning (approximation of video captioning using frames)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Load a frame from video (here using a sample image)
img_url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
image = Image.open(requests.get(img_url, stream=True).raw)

inputs = processor(image, return_tensors="pt")
out = model.generate(**inputs)
caption = processor.decode(out[0], skip_special_tokens=True)
print("Generated Caption:", caption)


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Generated Caption: a woman sitting on the beach with her dog


## Demo 3: Text Summarization

In [5]:
# Install first if needed:
# pip install transformers sentencepiece torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model + tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Input text
article = """
New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving court, Barrientos was arrested and charged with theft of service and criminal trespass for allegedly sneaking into the New York subway through an emergency exit, said Detective
Annette Markowski, a police spokeswoman. In total, Barrientos has been married 10 times, with nine of her marriages occurring between 1999 and 2002.
All occurred either in Westchester County, Long Island, New Jersey or the Bronx. She is believed to still be married to four men, and at one time, she was married to eight men at once, prosecutors say.
Prosecutors said the immigration scam involved some of her husbands, who filed for permanent residence status shortly after the marriages.
Any divorces happened only after such filings were approved. It was unclear whether any of the men will be prosecuted.
The case was referred to the Bronx District Attorney\'s Office by Immigration and Customs Enforcement and the Department of Homeland Security\'s
Investigation Division. Seven of the men are from so-called "red-flagged" countries, including Egypt, Turkey, Georgia, Pakistan and Mali.
Her eighth husband, Rashid Rajput, was deported in 2006 to his native Pakistan after an investigation by the Joint Terrorism Task Force.
"""

# Tokenize
inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

# Generate summary
summary_ids = model.generate(
    **inputs,
    max_length=200,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

# Decode summary
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:")
print(article)
print("\nSummary:")
print(summary_text)

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Original Text:

New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving co

## Demo 4: Sentiment Analysis

In [6]:
# pip install transformers

from transformers import pipeline

# Load sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Run on customer reviews
reviews = [
    "The product quality is amazing and delivery was quick!",
    "Terrible experience. The item arrived broken.",
    "Average quality, not worth the price."
]

for review in reviews:
    result = sentiment_analyzer(review)[0]
    print(f"Review: {review}")
    print(f"Sentiment: {result['label']} (Score: {result['score']:.2f})")
    print("-" * 50)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Review: The product quality is amazing and delivery was quick!
Sentiment: POSITIVE (Score: 1.00)
--------------------------------------------------
Review: Terrible experience. The item arrived broken.
Sentiment: NEGATIVE (Score: 1.00)
--------------------------------------------------
Review: Average quality, not worth the price.
Sentiment: NEGATIVE (Score: 1.00)
--------------------------------------------------
